In [ ]:
import os
import psycopg2

DB_CONFIG = {
    "dbname": "FACEPROJECT",
    "user": "postgres",
    "password": os.environ.get("DB_PASSWORD", ""),
    "host": "localhost",
    "port": 5432
}

def resolve_incident():
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()

    # 1. Fetch active dispatches
    cur.execute("""
        SELECT d.id, m.name, r.name, d.sighting_location, d.dispatched_at, m.id
        FROM rescue_dispatches d
        JOIN missing_persons m ON d.missing_person_id = m.id
        JOIN rescue_teams r ON d.rescue_team_id = r.id
        WHERE d.status = 'EN_ROUTE'
        ORDER BY d.dispatched_at DESC;
    """)
    active_dispatches = cur.fetchall()

    if not active_dispatches:
        print("[*] No pending or en-route dispatches.")
        return

    print("\n=== ACTIVE FIELD DISPATCHES ===")
    for row in active_dispatches:
        d_id, p_name, t_name, loc, dispatched_time, _ = row
        print(f"[{d_id}] Person: {p_name} | Team: {t_name} | Location: {loc} | Dispatched: {dispatched_time}")

    dispatch_id = input("\nEnter Dispatch ID to update: ").strip()
    
    print("\nSelect Outcome:")
    print("  [1] FOUND: Person successfully located and secured")
    print("  [2] NOT FOUND: Person moved before team arrived (Update Last Seen)")
    print("  [3] FALSE ALARM: Match was an incorrect bystander")
    outcome_choice = input("Choice (1/2/3): ").strip()

    notes = input("Enter responder notes / debrief: ").strip()

    # Get associated person and team IDs
    cur.execute("SELECT missing_person_id, rescue_team_id, sighting_location FROM rescue_dispatches WHERE id = %s;", (dispatch_id,))
    res = cur.fetchone()
    if not res:
        print("[!] Invalid Dispatch ID.")
        return
    person_id, team_id, sighting_loc = res

    if outcome_choice == "1":
        # 1. Person Found: close case and dispatch
        cur.execute("""
            UPDATE rescue_dispatches 
            SET status = 'VERIFIED_FOUND', responder_notes = %s, resolved_at = CURRENT_TIMESTAMP 
            WHERE id = %s;
        """, (notes, dispatch_id))

        cur.execute("""
            UPDATE missing_persons 
            SET case_status = 'RESCUED' 
            WHERE id = %s;
        """, (person_id,))

        print(f"\n[+] SUCCESS: Case #{person_id} marked as RESCUED. Camera alerts silenced.")

    elif outcome_choice == "2":
        # 2. Not found, but update last seen location
        cur.execute("""
            UPDATE rescue_dispatches 
            SET status = 'NOT_FOUND', responder_notes = %s, resolved_at = CURRENT_TIMESTAMP 
            WHERE id = %s;
        """, (notes, dispatch_id))

        cur.execute("""
            UPDATE missing_persons 
            SET last_seen_location = %s 
            WHERE id = %s;
        """, (f"Last CCTV Sighting: {sighting_loc}", person_id))

        print(f"\n[-] Updated: Person not located. Last seen location updated to '{sighting_loc}'. Case remains ACTIVE.")

    elif outcome_choice == "3":
        # 3. False alarm
        cur.execute("""
            UPDATE rescue_dispatches 
            SET status = 'FALSE_ALARM', responder_notes = %s, resolved_at = CURRENT_TIMESTAMP 
            WHERE id = %s;
        """, (notes, dispatch_id))

        print("\n[!] Logged as FALSE ALARM. Case remains ACTIVE.")

    # Free up the rescue team regardless of outcome
    cur.execute("UPDATE rescue_teams SET is_available = TRUE WHERE id = %s;", (team_id,))
    conn.commit()

    cur.close()
    conn.close()
    print("[+] Responder team has been set back to AVAILABLE.")


if __name__ == "__main__":
    resolve_incident()